In [ ]:
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

import pandas as pd
from matplotlib import pyplot as plt

load_dotenv()
os.chdir(os.getenv("ROOT"))

from notebooks import splits, taxonomy

In [ ]:
TWEETS_CSV = os.getenv("TWEETS_CSV")
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
SPLIT_MANIFEST_CSV = Path(os.getenv("SPLIT_MANIFEST_CSV"))
SPLIT_TRAIN_CSV = Path(os.getenv("SPLIT_TRAIN_CSV"))
SPLIT_TEST_CSV = Path(os.getenv("SPLIT_TEST_CSV"))

# The seed is fixed to 42 before any result exists; it is never tuned.
FROZEN_SEED = 42
if RANDOM_STATE != FROZEN_SEED and os.getenv("SPLIT_ALLOW_OTHER_SEED") != "1":
    raise ValueError(
        f"RANDOM_STATE must be {FROZEN_SEED} (got {RANDOM_STATE}); "
        "SPLIT_ALLOW_OTHER_SEED=1 is for tests only"
    )
if TEST_SIZE != 0.2 and os.getenv("SPLIT_ALLOW_OTHER_SEED") != "1":
    raise ValueError(
        f"TEST_SIZE must be 0.2 (got {TEST_SIZE}); SPLIT_ALLOW_OTHER_SEED=1 is for tests only"
    )

In [ ]:
corpus = pd.read_csv(filepath_or_buffer=TWEETS_CSV)
print(f"corpus rows: {len(corpus)}")
corpus.head()

In [ ]:
manifest = splits.build_split(corpus, seed=RANDOM_STATE, test_fraction=TEST_SIZE)
meta = manifest.attrs["metadata"]
print("manifest sha256:", meta["manifest_sha256"])
print("counts:", meta["counts"])

In [ ]:
id_col = meta["columns"]["id"]
annotated = splits.assign_split(corpus, manifest, id_col)
annotated = annotated.assign(_sort=annotated[id_col].astype(str)).sort_values("_sort", kind="mergesort").drop(columns="_sort")
columns = list(corpus.columns) + ["group", "inner_fold"]
train_out = annotated[annotated["split"] == "train"][columns]
test_out = annotated[annotated["split"] == "test"][columns]

for path in (SPLIT_MANIFEST_CSV, SPLIT_TRAIN_CSV, SPLIT_TEST_CSV):
    path.parent.mkdir(parents=True, exist_ok=True)
splits.write_manifest(manifest, SPLIT_MANIFEST_CSV)
train_out.to_csv(SPLIT_TRAIN_CSV, index=False)
test_out.to_csv(SPLIT_TEST_CSV, index=False)
len(train_out), len(test_out)

In [ ]:
# Integrity: re-read everything from disk and verify against the corpus.
corpus_again = pd.read_csv(filepath_or_buffer=TWEETS_CSV)
manifest_disk = splits.read_manifest(SPLIT_MANIFEST_CSV)
splits.verify_manifest(corpus_again, manifest_disk, expected_seed=RANDOM_STATE)

train_disk = pd.read_csv(SPLIT_TRAIN_CSV)
test_disk = pd.read_csv(SPLIT_TEST_CSV)
for name, frame, split in (("train", train_disk, "train"), ("test", test_disk, "test")):
    assert list(frame.columns) == columns, f"{name}: unexpected columns"
    ids = frame[id_col].astype(str)
    assert ids.is_monotonic_increasing and ids.is_unique, f"{name}: not sorted by unique row id"
    expected = manifest_disk[manifest_disk["split"] == split].sort_values("row_id")
    assert list(ids) == list(expected["row_id"]), f"{name}: row ids differ from the manifest"
    assert (frame["group"].astype(str).to_numpy() == expected["group"].to_numpy()).all(), f"{name}: groups differ"
    assert (frame["inner_fold"].to_numpy() == expected["inner_fold"].to_numpy()).all(), f"{name}: folds differ"
assert set(train_disk[id_col].astype(str)).isdisjoint(test_disk[id_col].astype(str))
assert len(train_disk) + len(test_disk) == len(corpus_again) == len(manifest_disk)
assert not set(train_disk["group"]) & set(test_disk["group"]), "a group spans train and test"
print("verify_manifest and file checks passed")
print("MANIFEST_SHA256 (pin downstream):", splits.manifest_sha256(manifest_disk))
print("seed:", manifest_disk.attrs["metadata"]["seed"], "| input sha256:", manifest_disk.attrs["metadata"]["input_sha256"])

In [ ]:
# Corpus and group descriptives.
rel = corpus["relevant"].map(taxonomy.as_bool)
print(f"corpus rows: {len(corpus)} | relevant: {int(rel.sum())} | irrelevant: {int((~rel).sum())}")
g = meta["grouping"]
c = meta["counts"]
print("grouping:", {k: v for k, v in g.items() if k != "merged_component_rows"})
print("merged component size -> count:", g.get("merged_component_rows"))
print(f"groups: {c['groups']} | duplicate groups: {c['duplicate_groups']} "
      f"({c['rows_in_duplicate_groups']} rows) | largest group: {c['largest_group']} rows")

wls = [taxonomy.watson_list(w) for w in corpus["watson"]]
signature = [
    (bool(r), tuple(sorted(taxonomy.top_labels(wl, r, "one_step"))))
    for r, wl in zip(rel, wls)
]
by_group = pd.DataFrame({"group": manifest.set_index("row_id").loc[corpus[id_col].astype(str), "group"].to_numpy(),
                         "sig": signature, "rel": rel.to_numpy()})
conflict_labels = by_group.groupby("group")["sig"].nunique().gt(1).sum()
conflict_relevance = by_group.groupby("group")["rel"].nunique().gt(1).sum()
print(f"groups with relevance or one-step label conflicts: {conflict_labels} | of which relevance conflicts: {conflict_relevance}")

print(f"train rows/groups: {c['train_rows']}/{c['train_groups']} | test rows/groups: {c['test_rows']}/{c['test_groups']}")
print("inner-fold rows:", c["inner_fold_rows"], "| groups:", c["inner_fold_groups"])
_, ax = plt.subplots(figsize=(5, 3))
pd.Series(c["inner_fold_rows"]).plot(kind="bar", color="skyblue", ax=ax)
plt.title("Inner-fold rows (train)")
plt.xlabel("fold")
plt.show()

In [ ]:
# Per-label support by split (taxonomy labels; rows and groups).
pd.set_option("display.width", 200)
show = ["label", "total", "train", "test", "test_share", "test_dev", "max_fold_dev", "min_fold"]
two_step = splits.balance_report(corpus, manifest)
one_step_labels = [
    [f"one:{t}" for t in taxonomy.top_labels(wl, r, "one_step")] for r, wl in zip(rel, wls)
]
one_step = splits.balance_report(corpus, manifest, labels=one_step_labels)

print("Two-step: relevance, top level and sub levels")
for level in ("rows", "groups"):
    print(f"-- {level}")
    print(two_step[two_step["level"] == level][show].round(3).to_string(index=False))
print("One-step top level")
for level in ("rows", "groups"):
    print(f"-- {level}")
    print(one_step[one_step["level"] == level][show].round(3).to_string(index=False))

top = two_step[(two_step["level"] == "rows") & two_step["label"].str.startswith("top:")].set_index("label")
_, ax = plt.subplots(figsize=(8, 3))
top[["train", "test"]].plot(kind="bar", stacked=True, ax=ax, color=["skyblue", "orange"])
plt.title("Two-step top-level support (relevant rows)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()